# 01. Physics-Informed Energy Imputation & Scaling Uncertainty
**Objective:** Address the ~84% missing hardware/energy telemetry in Epoch AI's dataset by modeling $\log_{10}(\text{Facility Energy MWh}) \sim \log_{10}(\text{Training Compute FLOP})$ with **80% and 95% Prediction Intervals**.

In [ ]:
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
import plotly.graph_objects as go

df = pd.read_csv('../data/02_gold_exports/models_enriched_gold.csv')
valid = df.dropna(subset=['training_compute_flop', 'facility_total_energy_mwh']).copy()
valid = valid[(valid['training_compute_flop'] > 0) & (valid['facility_total_energy_mwh'] > 0)]

valid['log_flop'] = np.log10(valid['training_compute_flop'])
valid['log_energy'] = np.log10(valid['facility_total_energy_mwh'])
print(f'Total Gold models: {len(df)} | Physics-verified training runs: {len(valid)} ({len(valid)/len(df):.1%})')

In [ ]:
# Fit OLS Log-Linear Scaling Law with 80% and 95% Prediction Intervals
X = sm.add_constant(valid['log_flop'])
model = sm.OLS(valid['log_energy'], X).fit()
print(model.summary())

x_grid = np.linspace(valid['log_flop'].min(), valid['log_flop'].max(), 200)
X_pred = sm.add_constant(x_grid)
pred_95 = model.get_prediction(X_pred).summary_frame(alpha=0.05)
pred_80 = model.get_prediction(X_pred).summary_frame(alpha=0.20)

# Apply imputation to undisclosed models that report FLOPs
missing_energy = df[df['facility_total_energy_mwh'].isna() & (df['training_compute_flop'] > 0)].copy()
missing_energy['log_flop'] = np.log10(missing_energy['training_compute_flop'])
imp_pred = model.get_prediction(sm.add_constant(missing_energy['log_flop'])).summary_frame(alpha=0.05)
missing_energy['imputed_energy_mwh_p50'] = 10 ** imp_pred['mean']
missing_energy['imputed_energy_mwh_low95'] = 10 ** imp_pred['obs_ci_lower']
missing_energy['imputed_energy_mwh_high95'] = 10 ** imp_pred['obs_ci_upper']

print(f'Imputed energy bounds for {len(missing_energy)} undisclosed models.')
missing_energy[['model_name', 'organization', 'publication_year', 'imputed_energy_mwh_p50', 'imputed_energy_mwh_low95', 'imputed_energy_mwh_high95']].sort_values('imputed_energy_mwh_p50', ascending=False).head(10)

In [ ]:
# Executive Visualization: Chart 8 - Energy Scaling Law & Prediction Uncertainty
fig = go.Figure()
fig.add_trace(go.Scatter(x=x_grid, y=10**pred_95['obs_ci_upper'], mode='lines', line=dict(width=0), showlegend=False))
fig.add_trace(go.Scatter(x=x_grid, y=10**pred_95['obs_ci_lower'], mode='lines', line=dict(width=0), fill='tonexty', fillcolor='rgba(59, 130, 246, 0.12)', name='95% Prediction Interval'))
fig.add_trace(go.Scatter(x=x_grid, y=10**pred_80['obs_ci_upper'], mode='lines', line=dict(width=0), showlegend=False))
fig.add_trace(go.Scatter(x=x_grid, y=10**pred_80['obs_ci_lower'], mode='lines', line=dict(width=0), fill='tonexty', fillcolor='rgba(59, 130, 246, 0.25)', name='80% Prediction Interval'))
fig.add_trace(go.Scatter(x=x_grid, y=10**pred_95['mean'], mode='lines', line=dict(color='#2563EB', width=2.5), name=f'Log-Linear Fit (R^2 = {model.rsquared:.2f})'))
fig.add_trace(go.Scatter(
    x=valid['log_flop'], y=valid['facility_total_energy_mwh'], mode='markers',
    text=valid['model_name'], marker=dict(size=7, color='#0F172A', opacity=0.75),
    name='Disclosed Hardware Telemetry'
))
fig.update_layout(
    title='Physics-Informed Energy Scaling Law & Imputation Uncertainty Bands',
    xaxis_title='Training Compute (log10 FLOP)',
    yaxis_title='Facility Total Energy (MWh, PUE-Adjusted)',
    yaxis_type='log', template='plotly_white', width=1050, height=580
)
fig.write_image('../assets/charts/chart8_imputation_uncertainty.png', scale=2)
fig.show()